In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, ParameterGrid
import pickle
import os

# LR on BioClinicalBERT Embeddings

In [ ]:
def get_bootstrap_metrics(y_true, y_pred, n_bootstrap=10000, seed=42):
    """Calculates classification metrics (including balanced accuracy) and 95% CIs using bootstrapping."""
    np.random.seed(seed)
    n = len(y_true)
    metrics = {k: np.zeros(n_bootstrap) for k in ['bal_acc', 'prec', 'rec', 'spec', 'npv', 'f1']}

    for i in range(n_bootstrap):
        indices = np.random.randint(0, n, n)
        yt, yp = y_true[indices], y_pred[indices]

        tp = np.sum((yt == 1) & (yp == 1))
        tn = np.sum((yt == 0) & (yp == 0))
        fp = np.sum((yt == 0) & (yp == 1))
        fn = np.sum((yt == 1) & (yp == 0))

        metrics['prec'][i] = tp / (tp + fp) if (tp + fp) > 0 else 0
        metrics['rec'][i] = tp / (tp + fn) if (tp + fn) > 0 else 0
        metrics['spec'][i] = tn / (tn + fp) if (tn + fp) > 0 else 0
        metrics['npv'][i] = tn / (tn + fn) if (tn + fn) > 0 else 0

        p, r = metrics['prec'][i], metrics['rec'][i]
        metrics['f1'][i] = 2 * p * r / (p + r) if (p + r) > 0 else 0

        # Balanced Accuracy = (Sensitivity + Specificity) / 2
        metrics['bal_acc'][i] = (metrics['rec'][i] + metrics['spec'][i]) / 2

    def ci(data):
        return np.mean(data), np.percentile(data, 2.5), np.percentile(data, 97.5)

    return {
        'balanced accuracy': ci(metrics['bal_acc']),
        'precision': ci(metrics['prec']),
        'recall': ci(metrics['rec']),
        'specificity': ci(metrics['spec']),
        'npv': ci(metrics['npv']),
        'f1': ci(metrics['f1'])
    }

def format_metrics(metrics_dict):
    lines = [f"{metric.title()}: {mean:.3f} (95% CI: {lower:.3f} - {upper:.3f})"
             for metric, (mean, lower, upper) in metrics_dict.items()]
    return "\n".join(lines)

def map_outcomes(y_true, y_pred):
    outcomes = []
    for yt, yp in zip(y_true, y_pred):
        if yt == 1 and yp == 1: outcomes.append("tp")
        elif yt == 0 and yp == 0: outcomes.append("tn")
        elif yt == 0 and yp == 1: outcomes.append("fp")
        else: outcomes.append("fn")
    return outcomes

In [ ]:
def train_logistic_regression(save_path, fold=5):
    """
    Tunes Logistic Regression via k-fold CV, retrains on full data,
    and evaluates on a holdout test set with bootstrapped metrics.
    """
    data_df = pd.read_pickle("../data/merged_data_embedded_pritamdekaS-PubMedBert-MS-MARCO_maxpool.pkl")
    X = np.stack(data_df["embedding"].to_numpy())
    y = data_df["label"].to_numpy()

    outdir = f"../data/models/{save_path}"
    os.makedirs(outdir, exist_ok=True)

    param_grid = {
        'C': [0.001, 0.01, 0.1, 1.0, 10.0],
        'class_weight': [None, 'balanced']
    }
    grid = list(ParameterGrid(param_grid))
    kf = KFold(n_splits=fold, shuffle=True, random_state=42)

    best_f1 = -1
    best_params = None

    print("--- Starting Grid Search for Logistic Regression ---")
    for params in grid:
        fold_f1s = []
        for train_idx, val_idx in kf.split(X, y):
            model = LogisticRegression(**params, max_iter=1000, random_state=42)
            model.fit(X[train_idx], y[train_idx])
            yp = model.predict(X[val_idx])

            tp = np.sum((y[val_idx] == 1) & (yp == 1))
            fp = np.sum((y[val_idx] == 0) & (yp == 1))
            fn = np.sum((y[val_idx] == 1) & (yp == 0))

            prec = tp / (tp + fp) if (tp + fp) > 0 else 0
            rec = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
            fold_f1s.append(f1)

        avg_f1 = np.mean(fold_f1s)
        print(f"Params: {params} | Avg F1: {avg_f1:.4f}")
        if avg_f1 > best_f1:
            best_f1 = avg_f1
            best_params = params

    print(f"\n[!] Best Params Found: {best_params} with Avg F1: {best_f1:.4f}")

    print(f"\n--- Running {fold}-fold CV with Best Params ---")
    for split_index, (train_index, test_index) in enumerate(kf.split(X, y)):
        model = LogisticRegression(**best_params, max_iter=1000, random_state=42)

        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]

        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

        metrics = get_bootstrap_metrics(y_test, predictions, n_bootstrap=10000, seed=42)
        formatted_metrics = format_metrics(metrics)

        si = f"Split {split_index}:"
        print(f"\n{si}")
        print(formatted_metrics)

        with open(f'{outdir}/split_{split_index}_performance.txt', 'w') as f:
            f.write(si + '\n' + formatted_metrics + '\n')

        with open(f"{outdir}/split_{split_index}_model.pkl", "wb") as f:
            pickle.dump(model, f)

        combined_df = data_df.iloc[test_index].copy()
        combined_df["prediction"] = predictions
        combined_df["outcome"] = map_outcomes(y_test, predictions)
        if "embedding" in combined_df.columns:
            combined_df = combined_df.drop(columns=["embedding"])
        combined_df.to_csv(f"{outdir}/split_{split_index}_result.csv", index=False)

    print("\n--- Retraining Best Model on Full Training Set ---")
    full_model = LogisticRegression(**best_params, max_iter=1000, random_state=42)
    full_model.fit(X, y)

    train_preds = full_model.predict(X)
    train_metrics = get_bootstrap_metrics(y, train_preds, n_bootstrap=10000, seed=42)

    print("\nFull Training Set Performance:")
    print(format_metrics(train_metrics))

    with open(f'{outdir}/full_train_performance.txt', 'w') as f:
        f.write("Full Training Set Performance:\n" + format_metrics(train_metrics) + '\n')

    with open(f"{outdir}/full_model.pkl", "wb") as f:
        pickle.dump(full_model, f)

    print("\n--- Evaluating on Holdout Test Set ---")
    test_df = pd.read_pickle("../data/test_merged_data_embedded_pritamdekaS-PubMedBert-MS-MARCO_maxpool.pkl")
    X_test_final = np.stack(test_df["embedding"].to_numpy())
    y_test_final = test_df["label"].to_numpy()

    test_preds = full_model.predict(X_test_final)
    test_metrics = get_bootstrap_metrics(y_test_final, test_preds, n_bootstrap=10000, seed=42)

    print("\nTest Set Performance:")
    print(format_metrics(test_metrics))

    with open(f'{outdir}/test_performance.txt', 'w') as f:
        f.write("Test Set Performance:\n" + format_metrics(test_metrics) + '\n')

    test_df["prediction"] = test_preds
    test_df["outcome"] = map_outcomes(y_test_final, test_preds)
    if "embedding" in test_df.columns:
        test_df = test_df.drop(columns=["embedding"])
    test_df.to_csv(f"{outdir}/test_result.csv", index=False)

    print("\nPipeline Complete!")

In [ ]:
train_logistic_regression("bioclinicalbert_lr_v4_PubMedBert-MS-MARCO-maxpool")